<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/simple_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

In [ ]:
!pip install dspy

In [ ]:
import dspy

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=2048)
dspy.configure(lm=lm)

In [ ]:
import pandas as pd

instructions_df = pd.read_excel('Dungeons-and-Dragons-Turn-Classification/instructions.xlsx')

In [ ]:
instructions_df

In [ ]:
prompts_df = instructions_df.drop_duplicates(subset=[0], inplace=False)

In [ ]:
prompts_df

In [ ]:
prompts_df = prompts_df.drop(columns=['Unnamed: 0'], inplace=False)

In [ ]:
prompts_df

In [ ]:
prompts = []

for classes, instructions in prompts_df.values:
    examples = dspy.Example(classes=classes, instructions=instructions).with_inputs("classes", "instructions")
    prompts.append(examples)

In [ ]:
prompts

In [ ]:
from typing import Literal

class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['Gameplay Mechanic',
                      'Out-Of-Game Conversation',
                      'Worldbuilding',
                      'Strategising',
                      ] = dspy.OutputField()

In [ ]:
class ClassifyTurns(dspy.Module):
  def __init__(self):
    self.classifier = dspy.ChainOfThought(TurnClassifier, caching=False)

  def forward(self, context, question, **kwargs):
    prediction = self.classifier(context=context, question=question)
    return prediction

In [ ]:
classify = ClassifyTurns()
classify.load("Dungeons-and-Dragons-Turn-Classification/optimized_classifier.json")

Tools with Classifier

In [ ]:
import pandas as pd
import datetime

class PromptTool:
  """Tool for retrieving appropriate instruction from excel spreadsheet."""
  def __init__(self):
    self.prompts = prompts
    self.classify = classify

  def classify_input(self, context: str, query:str, user_id: str = "default_user") -> str:
    """Classify the user's input into a category of Dungeons & Dragons player behaviour,
    using the optimized classifier."""
    try:
      prompt_class = self.classify(context=context, question=query)
      return prompt_class
    except Exception as e:
      return 0


  def search_prompts(self, prompt_class: str, context: str, query: str, user_id: str = "default_user", limit: int=4) -> str:
    """Search for the relevant addition to the instruction prompt given a user's input"""
    try:
        prompt = self.prompts(prompt_class, query=query, user_id=user_id, limit=limit)
        if not prompt:
          return "No relevant prompt found"

        prompt_text = "Using the following prompt:\n"
        for i, prompt, in enumerate(prompts["instructions"]):
          prompt_text += f"{i}.{prompt['instructions']}\n"
          return prompt_text
    except Exception as e:
      return f"Error searching prompts: {str(e)}"

  def update_prompt(self, context: str, query: str, prompt_class: str, new_prompt: str) -> str:
    """Update the existing instruction prompt based on the classification of the user's input"""
    try:
      self.prompts.update(prompt_class, new_prompt)
      return f"Updated prompt with new instructions: {new_prompt}"
    except Exception as e:
      return f"Error updating instructions: {str(e)}"


Tools Without Classifier

In [ ]:
# import pandas as pd
# import datetime

# class PromptTool:
#   """Tool for retrieving appropriate instruction from excel spreadsheet."""
#   def __init__(self):
#     self.prompts = prompts


#   def search_prompts(self, prompt_class: str, context: str, query: str, user_id: str = "default_user", limit: int=4) -> str:
#     """Search for the relevant addition to the instruction prompt given a user's input"""
#     try:
#       prompt = self.prompts.search(prompt_class, query, user_id=user_id, limit=limit)
#       if not prompt:
#         return "No relevant prompt found"

#       prompt_text = "Using the following prompt:\n"
#       for i, prompt, in enumerate(prompts["instructions"]):
#         prompt_text += f"{i}.{prompt['instructions']}\n"
#         return prompt_text
#     except Exception as e:
#       return f"Error searching prompts: {str(e)}"

#   def update_prompt(self, context: str, query: str, prompt_class: str, new_prompt: str) -> str:
#     """Update the existing instruction prompt based on the classification of the user's input"""
#     try:
#       self.prompt.update(prompt_class, new_prompt)
#       return f"Updated prompt with new instructions: {new_prompt}"
#     except Exception as e:
#       return f"Error updating instructions: {str(e)}"

# def get_current_time() -> str:
#     """Get the current date and time."""
#     return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
class ChatbotSignature(dspy.Signature):

  instruct_prompt: str = dspy.InputField()
  scenario_context: str = dspy.InputField()
  user_input: str = dspy.InputField()
  response: str = dspy.OutputField()

class InstructedChatbot(dspy.Module):
  """A ReAct agent enhanced with a set of instructions based on Dungeons & Dragons
  player behaviour"""

  def __init__(self):
    super().__init__()
    self.prompt_tools = PromptTool()

    self.tools = [
        self.prompt_tools.classify_input,
        self.prompt_tools.search_prompts,
        self.prompt_tools.update_prompt,
    ]

    self.react = dspy.ReAct(
        signature=ChatbotSignature,
        tools = self.tools,
        max_iters=6,
    )

  def forward(self, instruct_prompt: str, scenario_context: str, user_input: str):
    """Process user input with prompt-aware reasoning"""
    return self.react(instruct_prompt=instruct_prompt, scenario_context=scenario_context, user_input=user_input)



In [ ]:
import time

def run_agent_demo():
  """Demonstration of prompt-enhanced ReAct agent."""
  lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434',
               api_key='', max_tokens=2048)
  dspy.configure(lm=lm)

  agent = InstructedChatbot()

  print("Prompt Enhanced ReAct Agent Demo")
  print("="*50)

  instruct_prompt = [
        'The instructions for how to play the role of a D&D player are as '
        'follows. This is a short scenario in which you '
        'play the role of a character named Bob. This scenario '
        'is structured as a Dungeons & Dragons game. '
        'The goal is to be consistent, but creative. It is '
        'important to play the role of a Dungeons & Dragons player as '
        'accurately as possible, i.e., by responding in ways that you think '
        'it is likely a player would respond, and taking '
        'into account all information that you have. '
        'It is important that you collaborate with with the user who is the '
        'other player, on the task at hand.'
        'Always use first-person limited perspective.'


  ]

  scenario_context = [
  # 'This D&D short scenario specifically concerns a rat infestation.',
  # 'It is set in the craft brewery Company and is in dire need of help.',
  # 'Alice and Bob are here to sort out a RAT INFESTATION in the brewery\'s BASEMENT.',
  # 'For the duration of the scenario, only Alice, Bob and the brewery owner Charlie are in the brewery',
  # 'At the beginning of this adventure Alice and Bob',
  # 'meet in the Wizard Tower Brewing Company. These two adventurers',
  # 'DO NOT know each other AT FIRST and need to get to know each other.',
  # 'The Charlie hands out pints of Ale to Alice and Bob as they get to know each other.',
  "Charlie: I'm the owner of this brewery, do you not have any questions for me?"
  "Alice: So I'm a dwarf barbarian, I think we should just charge into the basement!"
  "Charlie: That's one thing you could try, you are the professional after all, but I would advise against it."
  ]

  # conversations = [
  #     "I'm Alice and I am a dwarf barbarian.",
  #     "As a dwarf barbarian I think we should take an active approach to solving this rat infestation.",
  #     "I want to head down to the basement of this brewery and find where the rats are coming from.",
  #     "How do you think we should approach this?"
  # ]

  # for i, user_input in enumerate(conversations, 1):
  #   print(f"\n User: {user_input}")

  try:
      for i in range(1):
        user_input = input()
        response = agent(instruct_prompt=instruct_prompt, scenario_context=scenario_context, user_input=user_input)
        print(f"Agent; {response.response}")
      time.sleep(1)

  except Exception as e:
      print(f"Error: {e}")

if __name__== "__main__":
  run_agent_demo()




In [ ]:
dspy.inspect_history()